# <PROJECT> — v1

**What this produces.** <one-line plain-language description a PM could read.>

**Inputs / method.** <key source tables; the headline method.>

Section map:
`0` config & helpers · `1` schema map & tunables · `2` schema verify + freshness ·
`3` eligibility (server-side) · `4` signals · `5` assemble · `6` normalize/score ·
`7` sanity & validation · `8` write-back (disabled)

> Golden rules: config is the single source of truth (§1) · verify + freshness
> before heavy work (§2) · ClickHouse-first, recompute from raw when stale ·
> memory-safe (scratch table + spill + flipped joins) · cache heavy queries ·
> scoring is whole-population (never batch percentiles) · writes disabled by default.

## 0 · Imports, config & connectivity

In [ ]:
import hashlib
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from clickhouse_driver import Client
from scipy.stats import spearmanr

# ─── Warehouse registry ──────────────────────────────────────────────────────
# Chartmetric runs SEPARATE ClickHouse Cloud warehouses. They are distinct
# clusters and NO account holds the REMOTE grant, so you CANNOT join across them
# in one query. Pick one per notebook.
#   rw-standard  — the music warehouse (chartmetric_analytics / _raw_data)
#   vert         — the new-verticals warehouse (new_vertical): athletes, brands
#
# Credential env vars differ by environment, so each entry lists candidates in
# priority order and the resolver tries each verbatim, upper- and lower-cased:
#   in the docker Jupyter (`make jupyter`)  CH_USER / CH_PASSWORD and
#       CH_VERT_USER / CH_VERT_PASSWORD are forwarded by docker-compose
#   with devin-secrets.env sourced         clickhouse_* / clickhouse_newverticals_*
# Host falls back to a per-warehouse default, because CLICKHOUSE_VERT_HOST is a
# devin-secrets.env name and is NOT forwarded into the Jupyter container.
CH_WAREHOUSES = {
    "rw-standard": {
        "host": ("CLICKHOUSE_HOST", "clickhouse_host"),
        "host_default": "grkyl47mbo.us-west-2.aws.clickhouse.cloud",
        "user": ("CH_USER", "clickhouse_user"),
        "password": ("CH_PASSWORD", "clickhouse_password"),
        "expect_db": "chartmetric_analytics",
    },
    "vert": {
        "host": ("CLICKHOUSE_VERT_HOST", "clickhouse_newverticals_host"),
        "host_default": "j1ez4a7j4k.us-west-2.aws.clickhouse.cloud",
        "user": ("CH_VERT_USER", "clickhouse_newverticals_user"),
        "password": ("CH_VERT_PASSWORD", "clickhouse_newverticals_password"),
        "expect_db": "new_vertical",
    },
}
WAREHOUSE = "rw-standard"   # <-- the ONLY place the target warehouse is named
CH_NATIVE_PORT = 9440       # clickhouse_driver native TLS (HTTPS would be 8443)


def _env(names, default=None):
    """First present env var among `names`, trying each verbatim/upper/lower."""
    for name in names:
        for cand in (name, name.upper(), name.lower()):
            val = os.environ.get(cand)
            if val:
                return val
    if default is not None:
        return default
    raise KeyError(
        f"none of {names} found in the environment — source "
        "~/code/chartmetric/devin-secrets.env, or run inside `make jupyter`")


def ch_config(warehouse=None):
    """Resolve {host, user, password, port} for a warehouse from the registry."""
    wh = warehouse or WAREHOUSE
    spec = CH_WAREHOUSES[wh]
    host = _env(spec["host"], spec["host_default"])
    # tolerate a full URL in the env var (the newverticals host is stored as one)
    host = host.split("://")[-1].split("/")[0].split(":")[0]
    return {"warehouse": wh, "host": host, "user": _env(spec["user"]),
            "password": _env(spec["password"]), "port": CH_NATIVE_PORT,
            "expect_db": spec["expect_db"]}


def ch_client(warehouse=None):
    """clickhouse_driver Client for the named warehouse.

    Builds the Client directly rather than going through
    data_utils.clickhouse_access.clickhouse_connect(). That helper does support
    `service="vert"` since data-utils 1.19.0, but OLDER versions SILENTLY ignore
    the argument and fall back to rw-standard — where `new_vertical` does not
    exist. A silent wrong-warehouse connection is the worst failure mode here,
    hence the explicit client plus the database assert below."""
    cfg = ch_config(warehouse)
    return Client(host=cfg["host"], user=cfg["user"], password=cfg["password"],
                  port=cfg["port"], secure=True, send_receive_timeout=3300)


# Memory-safety settings applied to EVERY query (spill instead of OOM). The
# vert warehouse is shared and small; queries here are deliberately lean.
CH_SETTINGS = {
    "allow_experimental_analyzer": 1,
    "memory_usage_overcommit_max_wait_microseconds": 60_000_000,
    "max_bytes_before_external_group_by": 4_000_000_000,
    "max_bytes_before_external_sort": 4_000_000_000,
    "join_algorithm": "auto",
    "max_threads": 8,
}

# ─── local parquet cache (keyed by hash of warehouse + SQL) ──────────────────
CACHE_DIR, CACHE_VERSION = "./_dscache", "v1"     # add _dscache/ to .gitignore
os.makedirs(CACHE_DIR, exist_ok=True)


def _cache_path(sql, warehouse):
    key = hashlib.sha1((CACHE_VERSION + (warehouse or WAREHOUSE) + sql).encode()).hexdigest()[:16]
    return os.path.join(CACHE_DIR, f"{key}.parquet")


def query_df(sql, cache=True, warehouse=None):
    """Run SQL -> DataFrame. Caches to parquet by hash(warehouse + sql); pass
    cache=False for cheap/volatile queries (freshness) or to force a refresh."""
    path = _cache_path(sql, warehouse)
    if cache and os.path.exists(path):
        return pd.read_parquet(path)
    con = ch_client(warehouse)
    try:
        rows, col_types = con.execute(sql, with_column_types=True, settings=CH_SETTINGS)
    finally:
        con.disconnect()
    df = pd.DataFrame(rows, columns=[n.split(".")[-1] for n, _ in col_types])
    if cache:
        df.to_parquet(path)
    return df


def run_ch(sql, warehouse=None):
    """DDL / statements with no result set."""
    con = ch_client(warehouse)
    try:
        con.execute(sql, settings=CH_SETTINGS)
    finally:
        con.disconnect()


_cfg = ch_config()
_ping = query_df("SELECT version() AS v, currentUser() AS u, today() AS today", cache=False)
print(f"warehouse: {_cfg['warehouse']}  host: {_cfg['host']}:{_cfg['port']}")
print(f"server: {_ping['v'].iloc[0]}  user: {_ping['u'].iloc[0]}  today: {_ping['today'].iloc[0]}")

# Assert we are on the warehouse we think we are. Both warehouses answer
# `SELECT 1` happily, so a connectivity check alone cannot tell them apart — and
# an accidental rw-standard connection would fail later with a confusing "table
# does not exist" for every athlete table.
_dbs = set(query_df("SHOW DATABASES", cache=False).iloc[:, 0])
assert _cfg["expect_db"] in _dbs, (
    f"connected to {_cfg['host']} but '{_cfg['expect_db']}' is not visible "
    f"(saw: {sorted(_dbs)}) — wrong warehouse, or this user lacks the grant")
print(f"database check OK: '{_cfg['expect_db']}' is visible")

## 1 · Schema map & tunables

The **single source of truth** — the only cell to edit to re-point data or change
behavior. Every table has a one-line schema comment (key cols, types, gotchas).

In [ ]:
# ─── databases ───────────────────────────────────────────────────────────────
DB_ANALYTICS = "chartmetric_analytics"
DB_RAW       = "chartmetric_raw_data"
DB_SCRATCH   = "chartmetric_test"                 # a DB you can CREATE TABLE in
SCRATCH      = f"{DB_SCRATCH}.<project>_eligible"  # rebuilt every run

TABLES = {
    # short name : fully-qualified            # key columns / types / GOTCHAS
    "artist_cache": f"{DB_ANALYTICS}.cm_artist_cache",   # id(Int32), sp_monthly_listeners/sp_followers Nullable(Int32)
    # "some_stat":  f"{DB_RAW}.some_stat",               # CUMULATIVE -> per-entity value = argMax(metric, timestp)
    # "some_repl":  f"{DB_RAW}.some_repl",               # ReplacingMergeTree -> use FINAL
}

# ─── freshness sentinels (label, table, date column) — checked in §2 ─────────
SOURCES = [
    ("artist_cache", TABLES["artist_cache"], "modified_at"),
    # ("some_stat",  TABLES["some_stat"],    "timestp"),
]
STALE_DAYS = 14                                   # warn if a source lags today() by more than this

# ─── eligibility gates ───────────────────────────────────────────────────────
MIN_MONTHLY_LISTENERS = 1000
MIN_SP_FOLLOWERS      = 500

# ─── windows / knobs (document units + today()-relative intent) ──────────────
WINDOW_DAYS = 90

# ─── normalization ───────────────────────────────────────────────────────────
N_SIZE_BINS = 12                                  # peer bins over log10(size); whole-population

## 2 · Schema verification & freshness (fail loudly, fail early)

Assert every dependency's columns exist, and print `max(date)` per source. A
missing column fails here, not mid-scan; a stale source is visible, not silent.

In [ ]:
c = ch_client()
try:
    # ── column asserts ───────────────────────────────────────────────────────
    CHECKS = {
        TABLES["artist_cache"]: ("id", "sp_monthly_listeners", "sp_followers"),
        # TABLES["some_stat"]:  ("entity", "metric", "timestp"),
    }
    for fq, needed in CHECKS.items():
        db, tbl = fq.split(".", 1)
        cols = [r[0] for r in c.execute(
            f"SELECT name FROM system.columns WHERE database='{db}' AND table='{tbl}'")]
        for col in needed:
            assert col in cols, f"'{col}' missing on {fq}"
    print("schema verification passed")

    # ── freshness sentinels (live, never cached) ─────────────────────────────
    today = pd.Timestamp.today().normalize()
    for label, fq, datecol in SOURCES:
        mx = c.execute(f"SELECT max({datecol}) FROM {fq}")[0][0]
        lag = (today - pd.Timestamp(mx)).days
        print(f"{label:24s} max={mx}  ({lag}d old){'   <-- STALE' if lag > STALE_DAYS else ''}")
finally:
    c.disconnect()

## 3 · Eligibility — materialized server-side

Compute the candidate population once, server-side, as a small scratch table.
Downstream queries filter with `IN {ELIG}` — a server-side hash-set test — so no
id list travels through Python or bloats SQL text (the classic OOM).

In [ ]:
run_ch(f"""
CREATE OR REPLACE TABLE {SCRATCH}
ENGINE = MergeTree ORDER BY id AS
SELECT id, sp_followers, sp_monthly_listeners
FROM {TABLES['artist_cache']}
WHERE sp_monthly_listeners >= {MIN_MONTHLY_LISTENERS}
  AND sp_followers         >= {MIN_SP_FOLLOWERS}
""")

ELIG = f"(SELECT id FROM {SCRATCH})"     # every heavy query: WHERE <key> IN {ELIG}

pop = query_df(f"SELECT id, sp_followers, sp_monthly_listeners FROM {SCRATCH}", cache=False)
assert len(pop) > 0, "no eligible rows — check gates in §1 / scratch permissions"
print(f"eligible population: {len(pop):,}")
pop.head()

## 4 · Signals / features

One cached query per signal. Guard data quality explicitly; state whether a
missing value is a genuine zero or unknown (NaN).

In [ ]:
# EXAMPLE signal — replace with real logic. Note the ELIG filter + guards.
sig = query_df(f"""
    SELECT id AS id, count() AS n_obs
    FROM {TABLES['artist_cache']}
    WHERE id IN {ELIG}
    GROUP BY id
""")
for col in ("n_obs",):
    sig[col] = pd.to_numeric(sig[col], errors="coerce")
print(f"signal rows: {len(sig):,}")
sig.head()

## 5 · Assemble & derive

In [ ]:
df = pop.copy()
for part in (sig,):                              # add each signal DataFrame here
    df = df.merge(part, on="id", how="left")

for col in ("sp_followers", "sp_monthly_listeners"):   # coerce object-dtype numerics once
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["log_size"] = np.log10(df["sp_monthly_listeners"].clip(lower=1))
df["size_bin"] = pd.qcut(df["log_size"], q=N_SIZE_BINS, labels=False, duplicates="drop")
print(df.groupby("size_bin")["sp_monthly_listeners"].agg(["count", "min", "max"]))

## 6 · Normalize / score

> Whole-population by construction — percentiles/bins/medians span all rows at
> once. NEVER batch this step. For zero-heavy signals use plain within-group
> percentile rank (average ties), not zero-pinning (see the skill's validation
> reference).

In [ ]:
def norm_within(s: pd.Series, bins: pd.Series) -> pd.Series:
    """0-100 percentile rank within each peer bin; NaN passes through; average ties."""
    s = pd.to_numeric(s, errors="coerce").astype("float64")
    return s.groupby(bins).transform(lambda g: g.rank(pct=True) * 100.0)

df["signal_norm"] = norm_within(df["n_obs"], df["size_bin"])
# compose your score from normalized signals here:
df["score"] = df["signal_norm"]                  # placeholder
df[["id", "score"]].sort_values("score", ascending=False).head(10)

## 7 · Sanity, validation & explain

In [ ]:
# distribution
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(df["score"].dropna(), bins=40); ax[0].set_title("score distribution")
df.groupby("size_bin")["score"].median().plot.bar(ax=ax[1])
ax[1].set_title("median score by size bin (want ~flat if size-neutral)")
plt.tight_layout(); plt.show()

# coverage
print("coverage:", {c: round(df[c].notna().mean(), 3) for c in ["n_obs", "score"]})

# expected correlation (size-neutral score -> want ~0)
r = spearmanr(df["score"], df["sp_monthly_listeners"], nan_policy="omit").correlation
print(f"Spearman(score, size) = {r:.3f}")

# leak localization: within-bin corr of each raw signal vs the confounder (want ~0)
def within_bin_rho(sig, bins, conf):
    zs, ws = [], []
    for b, idx in bins.groupby(bins).groups.items():
        m = sig.loc[idx].notna() & conf.loc[idx].notna()
        if m.sum() < 30: continue
        rho = spearmanr(sig.loc[idx][m], conf.loc[idx][m]).correlation
        if rho is not None and not np.isnan(rho):
            zs.append(np.arctanh(np.clip(rho, -0.999, 0.999))); ws.append(m.sum())
    return float(np.tanh(np.average(zs, weights=ws))) if zs else np.nan
print("within-bin rho (n_obs vs size):",
      round(within_bin_rho(df["n_obs"], df["size_bin"], df["log_size"]), 3))

# AS-OF validation stub: when replacing a stored metric, reproduce it as-of the
# stored table's frozen date and confirm your recompute matches (see skill ref).

## 8 · Write-back (DISABLED — review §7 first)

Enable only after the sanity checks look right. Ships fully commented-out.

In [ ]:
# ─── DISABLED — uncomment to persist after review ───────────────────────────
# out = df[["id", "score"]].copy()
# out["timestp"] = pd.Timestamp.utcnow().date()
# run_ch(f"""
#     CREATE TABLE IF NOT EXISTS {DB_ANALYTICS}.<project>_score (
#         id Int64, score Float64, timestp Date
#     ) ENGINE = MergeTree ORDER BY (id, timestp)
# """)
# con = ch_client()
# try:
#     con.execute(f"INSERT INTO {DB_ANALYTICS}.<project>_score VALUES", out.to_dict("records"))
# finally:
#     con.disconnect()
#
# optional scratch cleanup:
# run_ch(f"DROP TABLE IF EXISTS {SCRATCH}")
print("write-back disabled")